# DJ 30 Trading BOT


In this project, we try to build a trading Bot which rely trading decicion on a Machine Learning of 3 main components that we beleive that are important on determining trading strategy and timing. These components are Fundamental, Technical, and Sentiment In each component, we biuld buy/sell signals based on the represent indicators. Benefit from haveing Bot trading DJ 30 is to rely decision on these signals we have from the core components and reduce emotion from determining strategy and timing to get in and out of DJ 30. 

DJ 30 are AXP, AMGN, AAPL, BA, CAT, CSCO, CVX, GS, HD, HON, IBM, INTC, JNJ, KO, JPM, MCD, MMM, MRK, MSFT, NKE, PG, TRV, UNH, CRM, VZ, V, WBA, WMT, DIS, DOW

In [ ]:
#Import libraries
import numpy as np
import pandas as pd
#import ta
import os
import ffn as ffn
import bt

from datetime import datetime, timedelta

import hvplot.pandas
import yfinance as yf
import matplotlib.pyplot as plt
plt.style.use
%matplotlib inline
print("Libraries imported")

## Functions

In [ ]:
#Function for building data of each stock in Long horizon coverage
def build_input_long(ticker):
    stocks = ticker
    end_date = datetime.now() - timedelta(3)
    start_date = end_date - timedelta(365*1)
    data = yf.download(stocks, start=start_date, end=end_date)
    data = data['Close'].dropna()
    return_df = data.pct_change()
    return_df = return_df.dropna()
    filter_fundamental_df  = fundamental_df[fundamental_df.Ticker.isin([ticker])]
    filter_rsi_df  = rsi_df[rsi_df.Ticker.isin([ticker])]
    filter_sentiment_df  = sentiment_df[sentiment_df.Ticker.isin([ticker])]
    df= pd.concat([return_df,filter_fundamental_df,filter_rsi_df,filter_sentiment_df], axis=1, join='outer')
    df.drop('Ticker', axis=1, inplace=True)
    df = df[df['Close'].notna()]
    df['Positive_Return'] = np.where(df['Close'] > 0, 1.0, 0.0)
    return df

In [ ]:
#Function for building data of each stock in Short horizon coverage
def build_input_short(ticker):
    stocks = ticker
    end_date = datetime.now() - timedelta(3)
    start_date = end_date - timedelta(33)
    data = yf.download(stocks, start=start_date, end=end_date)
    data = data['Close'].dropna()
    return_df = data.pct_change()
    return_df = return_df.dropna()
    filter_fundamental_df  = fundamental_df[fundamental_df.Ticker.isin([ticker])]
    filter_rsi_df  = rsi_df[rsi_df.Ticker.isin([ticker])]
    filter_sentiment_df  = sentiment_df[sentiment_df.Ticker.isin([ticker])]
    df= pd.concat([return_df,filter_fundamental_df,filter_rsi_df,filter_sentiment_df], axis=1, join='outer')
    df.drop('Ticker', axis=1, inplace=True)
    df = df[df['Close'].notna()]
    df['Positive_Return'] = np.where(df['Close'] > 0, 1.0, 0.0)
    return df

In [ ]:
#Data Preparation Function (Long Horizon)
#x_var_list_long = ["ROE_Sig", "OpMargin_Sig","DebtEq_Sig","Fscore_Sig","RSI"]

def X_train_long(df):
    x_var_list_long = ["ROE_Sig", "OpMargin_Sig","DebtEq_Sig","Fscore_Sig","RSI"]
    X = df[x_var_list_long].shift(1)
    split = int(0.7 * len(df.Close))
    X_train = X[: split].dropna()   
    return X_train

def X_test_long(df):
    x_var_list_long = ["ROE_Sig", "OpMargin_Sig","DebtEq_Sig","Fscore_Sig","RSI"]
    split = int(0.7 * len(df.Close))
    X = df[x_var_list_long].shift(1)
    X_test = X[split:].dropna()
    return X_test


def y_train_long(df):
    y = df["Positive_Return"]
    split = int(0.7 * len(df.Close))
    y_train = y[1: split].dropna()
    return y_train

def y_test_long(df):
    y = df["Positive_Return"]
    split = int(0.7 * len(df.Close))
    y_test = y[split:-1].dropna()
    return y_test

In [ ]:
#Data Preparation Function (Short Horizon)
#x_var_list_short = ["ROE_Sig", "OpMargin_Sig","DebtEq_Sig","Fscore_Sig","RSI","Sentiment"]
def X_train_short(df):
    x_var_list_short = ["ROE_Sig", "OpMargin_Sig","DebtEq_Sig","Fscore_Sig","RSI","Sentiment"]
    X = df[x_var_list_short].shift(1)
    split = int(0.7 * len(df.Close))
    X_train = X[: split].dropna()   
    return X_train

def X_test_short(df):
    x_var_list_short = ["ROE_Sig", "OpMargin_Sig","DebtEq_Sig","Fscore_Sig","RSI","Sentiment"]
    X = df[x_var_list_short].shift(1)
    split = int(0.7 * len(df.Close))
    X_test = X[split:].dropna()
    return X_test

def y_train_short(df):
    y = df["Positive_Return"]
    split = int(0.7 * len(df.Close))
    y_train = y[1: split].dropna()
    return y_train

def y_test_short(df):
    y = df["Positive_Return"]
    split = int(0.7 * len(df.Close))
    y_test = y[split:-1].dropna()
    return y_test

In [ ]:
#ML function
def random_forest(X_train, X_test, y_train, y_test):
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.datasets import make_classification
    model = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=0)
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    predictions = model.predict(X_test)
    results = y_test.to_frame()
    results["Predicted_Value"] = predictions
    return results

## Features

We build signal data  from 3 main components which are..
1. Fundamental: ROE, Operation margin, Debt-to-Equity ratio, and Pietroskie F-score
2. Technical: Bollinger Band, RSI, MACD
3. Sentiment: News relevant to company name
Sources of these data are XXX, Yahoo Finance, and News API.
However, we can pull News API by just 1 month back bacause of limitation of using trial account. With this limitation, we plan to split the data  into long-horizon and short-horizon data sets. In thr longer-horizon data, the features used in ML model are just Fundamental and Technical. For short-horizon, all 3 main features will be included in the ML model.

In [ ]:
#Import cvs
#Fundamental
fundamental_df = pd.read_csv(r"C:\Users\khuli\OneDrive\Documents\BCS\Project2\Project-2\fundamental_signals_new.csv")
fundamental_df.set_index(pd.to_datetime(fundamental_df['Date'], infer_datetime_format=True), inplace=True)
fundamental_df.drop(columns=['Date'], inplace=True)


#Technical
#technical_df = pd.read_csv(r"C:\Users\khuli\OneDrive\Documents\BCS\Project2\technical_signal.csv")
rsi_df = pd.read_csv(r"C:\Users\khuli\OneDrive\Documents\BCS\Project2\rsi_signal.csv")
rsi_df.set_index(pd.to_datetime(rsi_df['Date'], infer_datetime_format=True), inplace=True)
rsi_df.drop(columns=['Date'], inplace=True)
#macd_df = pd.read_csv(r"C:\Users\khuli\OneDrive\Documents\BCS\Project2\rsi_signal.csv")

#Sentiment
sentiment_df = pd.read_csv(r"C:\Users\khuli\OneDrive\Documents\BCS\Project2\sentiment_signal.csv")
sentiment_df.set_index(pd.to_datetime(sentiment_df['Date'], infer_datetime_format=True), inplace=True)
sentiment_df.drop(columns=['Date'], inplace=True)


In [ ]:
fundamental_df.head()

In [ ]:
sentiment_df.head()

In [ ]:
rsi_df.head()

## DF management preparation

We split ML into (1) Long-horizon model and (2) Short horizon model because of limitation from getting sentiment signal

### DF for Longer horizon coverage

AXP, AMGN, AAPL, BA, CAT, CSCO, CVX, GS, HD, HON, IBM, INTC, JNJ, KO, JPM, MCD, MMM, MRK, MSFT, NKE, PG, TRV, UNH, CRM, VZ, V, WBA, WMT, DIS, DOW

In [ ]:
ticker = "MSFT"

In [ ]:
# df for Long horizon model
df = build_input_long(ticker)
df.tail(10)

### DF for Shorter horizon coverage

In [ ]:
# df for Short horizon model
df_2 = build_input_short(ticker)
df_2.sample(10)

## Split Data

### Split data for Longer horizon

In [ ]:
X_train = X_train_long(df)
X_test = X_test_long(df)
y_train = y_train_long(df)
y_test = y_test_long(df)

### Split data for Shorter horizon

In [ ]:
X_train_2 = X_train_short(df_2)
X_test_2 = X_test_short(df_2)
y_train_2 = y_train_short(df_2)
y_test_2 = y_test_short(df_2)

## Machine Learning

### Long-Horizon ML model, plot and evaluation

In [ ]:
results = random_forest(X_train, X_test, y_train, y_test)
plot = results[['Positive_Return', 'Predicted_Value']].tail(10).plot(kind="bar",title="Long-Horizon ML of "+ticker)
from sklearn.metrics import classification_report
print(classification_report(results.Positive_Return,results.Predicted_Value))

### Short-Horizon ML model, plot and evaluation

In [ ]:
results = random_forest(X_train_2, X_test_2, y_train_2, y_test_2)
plot = results[['Positive_Return', 'Predicted_Value']].tail(10).plot(kind="bar",title="Short-Horizon ML of " + ticker)
print(classification_report(results.Positive_Return,results.Predicted_Value))